# Test General Agent with RAG

This notebook tests the `general_agent` node which uses RAG (Retrieval Augmented Generation) to answer questions using context from Pinecone.

In [1]:
from sahiloan_chatbot.application.chat_service.workflow.nodes import Nodes
from sahiloan_chatbot.application.chat_service.workflow.state import ChatState
from sahiloan_chatbot.infrastructure.db import get_pinecone_index
import json

/Users/vishnum/Library/Caches/pypoetry/virtualenvs/sahiloan-chatbot-h6twZ8gO-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-26 17:44:14.487 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:48 - Looking for eval dataset at: /Users/vishnum/sahiloan-customer-chatbot/data/evals/intent_router.json
2026-01-26 17:44:14.489 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:61 - Loading 25 eval queries for intent caching...
2026-01-26 17:44:25.710 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:73 - ✅ Cached 25 intent router embeddings


## 1. Initialize Nodes

In [2]:
# Initialize the Nodes class (includes Pinecone and embeddings)
print("Initializing Nodes...")
nodes = Nodes()
print("✅ Nodes initialized successfully!")

# Verify Pinecone connection
index = get_pinecone_index()
stats = index.describe_index_stats()
print(f"\n📊 Pinecone Stats:")
print(f"   Total vectors: {stats['total_vector_count']}")
print(f"   Dimension: {stats['dimension']}")

Initializing Nodes...
✅ Nodes initialized successfully!

📊 Pinecone Stats:
   Total vectors: 25
   Dimension: 1024


## 2. Test Single Query

In [3]:
# Create a test query
query = "How is Sahiloan different from banks?"

# Create state with the query
state = ChatState(
    messages=[{"role": "user", "content": query}]
)

print(f"🔍 Query: {query}")
print("\n" + "="*80)

# Run the general agent
result_state = nodes.general_agent(state)

# Extract the response
response = result_state["messages"][-1]["content"]

print("\n💬 Response:")
print(response)
print("\n" + "="*80)
print(f"✅ Route to: {result_state.get('route_to')}")

2026-01-26 16:54:44.171 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:54:44.172 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: How is Sahiloan different from banks?
2026-01-26 16:54:44.172 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


🔍 Query: How is Sahiloan different from banks?



2026-01-26 16:54:45.735 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 1 relevant chunks
2026-01-26 16:54:47.408 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 1 | latency: 3237.45ms | llm: gpt-4o-mini



💬 Response:
Sahiloan is different from banks in that we work for you, prioritizing your financial well-being rather than meeting monthly targets like bank relationship managers do. Our focus is on providing honest guidance without hidden agendas or urgency tactics, ensuring you receive clear and trustworthy advice for one of the biggest financial decisions of your life.

✅ Route to: end


## 3. Inspect Retrieved Context (Manual Check)

In [ ]:
# Let's manually check what context would be retrieved
query = "How is Sahiloan different from banks?"

# Generate embedding
query_embedding = nodes.embeddings.embed_query(query)

# Search Pinecone
results = nodes.pinecone_index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

print(f"🔍 Query: {query}\n")
print(f"Found {len(results['matches'])} results:\n")
print("="*80)

for i, match in enumerate(results['matches'], 1):
    score = match['score']
    text = match['metadata']['text']
    filename = match['metadata'].get('filename', 'unknown')
    
    print(f"\nResult {i}:")
    print(f"  Score: {score:.4f}")
    print(f"  Source: {filename}")
    print(f"  Will be used: {'✅ Yes' if score > 0.7 else '❌ No (score too low)'}")
    print(f"  Content preview:\n  {text[:200]}...")
    print("-"*80)

## 4. Test Multiple Queries

In [ ]:
# Test with multiple questions
test_queries = [
    "What is the difference between Home Loan and LAP?",
    "How is EMI calculated?",
    "Is Sahiloan free to use?",
    "What credit score do I need?",
    "How long does the loan process take?"
]

print("="*80)
print("TESTING MULTIPLE QUERIES")
print("="*80)

for i, query in enumerate(test_queries, 1):
    print(f"\n{i}. Query: {query}")
    print("-"*80)
    
    # Create state
    state = ChatState(
        messages=[{"role": "user", "content": query}]
    )
    
    # Get response
    result_state = nodes.general_agent(state)
    response = result_state["messages"][-1]["content"]
    
    # Print response
    print(f"Response: {response[:300]}...")
    print()

## 5. Test Edge Cases

In [4]:
# Test edge cases
edge_cases = [
    "What's the weather today?",  # Unrelated question
    "Tell me about cryptocurrency loans",  # Not in knowledge base
    "",  # Empty query
    "Hello",  # Simple greeting
]

print("="*80)
print("TESTING EDGE CASES")
print("="*80)

for query in edge_cases:
    if not query:
        query_display = "[EMPTY]"
    else:
        query_display = query
    
    print(f"\n🔍 Query: {query_display}")
    print("-"*80)
    
    try:
        state = ChatState(
            messages=[{"role": "user", "content": query}]
        )
        
        result_state = nodes.general_agent(state)
        response = result_state["messages"][-1]["content"]
        
        print(f"Response: {response[:200]}...")
    except Exception as e:
        print(f"❌ Error: {e}")
    
    print()

2026-01-26 16:55:22.999 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:55:23.001 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: What's the weather today?
2026-01-26 16:55:23.001 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


TESTING EDGE CASES

🔍 Query: What's the weather today?
--------------------------------------------------------------------------------


2026-01-26 16:55:23.783 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 0 relevant chunks
2026-01-26 16:55:23.783 | WARNING  | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:114 - No relevant context found in Pinecone
2026-01-26 16:55:25.990 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 0 | latency: 2991.53ms | llm: gpt-4o-mini
2026-01-26 16:55:25.990 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:55:25.991 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: Tell me about cryptocurrency loans
2026-01-26 16:55:25.992 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


Response: I'm sorry, but I don't have access to current weather information. I recommend checking a reliable weather website or app for the most accurate updates on today's weather. If you have any questions ab...


🔍 Query: Tell me about cryptocurrency loans
--------------------------------------------------------------------------------


2026-01-26 16:55:26.852 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 0 relevant chunks
2026-01-26 16:55:26.853 | WARNING  | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:114 - No relevant context found in Pinecone
2026-01-26 16:55:30.972 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 0 | latency: 4981.07ms | llm: gpt-4o-mini
2026-01-26 16:55:30.976 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:55:30.976 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: 
2026-01-26 16:55:30.977 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


Response: I don't have specific information about cryptocurrency loans in our knowledge base. However, I can provide a general overview. 

Cryptocurrency loans typically allow you to borrow funds using your cry...


🔍 Query: [EMPTY]
--------------------------------------------------------------------------------


2026-01-26 16:55:31.664 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 0 relevant chunks
2026-01-26 16:55:31.665 | WARNING  | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:114 - No relevant context found in Pinecone
2026-01-26 16:55:33.776 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 0 | latency: 2800.49ms | llm: gpt-4o-mini
2026-01-26 16:55:33.777 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:55:33.777 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: Hello
2026-01-26 16:55:33.778 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


Response: I'm here to help! If you have any questions about loans, our advisory services, or anything related to Sahiloan, please feel free to ask. While I may not have specific information at the moment, I can...


🔍 Query: Hello
--------------------------------------------------------------------------------


2026-01-26 16:55:34.511 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 0 relevant chunks
2026-01-26 16:55:34.512 | WARNING  | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:114 - No relevant context found in Pinecone
2026-01-26 16:55:36.085 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 0 | latency: 2308.01ms | llm: gpt-4o-mini


Response: Hello! How can I assist you today? If you have any questions about loans or our services at Sahiloan, feel free to ask!...



## 6. Interactive Testing

In [5]:
# Interactive: Change this query and run the cell
custom_query = "Can Sahiloan help me reduce my interest rate?"

print(f"🔍 Query: {custom_query}")
print("\n" + "="*80)
print("STEP-BY-STEP EXECUTION")
print("="*80)

# Step 1: Retrieve context
print("\n1️⃣ Retrieving context from Pinecone...")
query_embedding = nodes.embeddings.embed_query(custom_query)
results = nodes.pinecone_index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

relevant_count = sum(1 for m in results['matches'] if m['score'] > 0.7)
print(f"   Found {len(results['matches'])} results, {relevant_count} relevant (score > 0.7)")

# Show top result
if results['matches']:
    top = results['matches'][0]
    print(f"\n   Top match:")
    print(f"   - Score: {top['score']:.4f}")
    print(f"   - Source: {top['metadata'].get('filename', 'unknown')}")
    print(f"   - Preview: {top['metadata']['text'][:150]}...")

# Step 2: Generate response
print("\n2️⃣ Generating response with LLM...")
state = ChatState(
    messages=[{"role": "user", "content": custom_query}]
)
result_state = nodes.general_agent(state)

# Step 3: Display response
print("\n3️⃣ Final Response:")
print("="*80)
response = result_state["messages"][-1]["content"]
print(response)
print("="*80)

🔍 Query: Can Sahiloan help me reduce my interest rate?

STEP-BY-STEP EXECUTION

1️⃣ Retrieving context from Pinecone...


2026-01-26 16:57:00.340 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:57:00.341 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: Can Sahiloan help me reduce my interest rate?
2026-01-26 16:57:00.341 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


   Found 5 results, 1 relevant (score > 0.7)

   Top match:
   - Score: 0.7908
   - Source: sahiloan.md
   - Preview: ### Can Sahiloan help me reduce my interest rate?

Yes. We help you:
* Compare lender-wise interest structures
* Avoid teaser traps
* Explore balance ...

2️⃣ Generating response with LLM...


2026-01-26 16:57:00.967 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 1 relevant chunks
2026-01-26 16:57:03.553 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 1 | latency: 3212.79ms | llm: gpt-4o-mini



3️⃣ Final Response:
Yes, Sahiloan can help you reduce your interest rate. We assist you in comparing lender-wise interest structures, avoiding teaser traps, and exploring options like balance transfers or renegotiation. Even small rate differences can lead to significant savings over time.


## 7. Compare With and Without Context

In [6]:
# Compare response WITH context vs WITHOUT context
from langchain_core.messages import SystemMessage, HumanMessage

query = "Is Sahiloan free for customers?"

print("="*80)
print(f"Query: {query}")
print("="*80)

# WITH CONTEXT (using RAG)
print("\n✅ WITH CONTEXT (RAG - General Agent):")
print("-"*80)
state = ChatState(messages=[{"role": "user", "content": query}])
result = nodes.general_agent(state)
print(result["messages"][-1]["content"])

# WITHOUT CONTEXT (direct LLM)
print("\n\n❌ WITHOUT CONTEXT (Direct LLM - No RAG):")
print("-"*80)
llm_fact = nodes.llm_factory.get_gpt_4o_mini()
messages = [
    SystemMessage(content="You are a helpful assistant. Answer the user's question."),
    HumanMessage(content=query)
]
direct_response = llm_fact["llm"].invoke(messages)
print(direct_response.content)

print("\n" + "="*80)
print("💡 Notice how the RAG version has specific, accurate information!")
print("="*80)

2026-01-26 16:57:37.754 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:57:37.755 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: Is Sahiloan free for customers?
2026-01-26 16:57:37.756 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


Query: Is Sahiloan free for customers?

✅ WITH CONTEXT (RAG - General Agent):
--------------------------------------------------------------------------------


2026-01-26 16:57:39.734 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 0 relevant chunks
2026-01-26 16:57:39.735 | WARNING  | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:114 - No relevant context found in Pinecone
2026-01-26 16:57:41.372 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 0 | latency: 3618.81ms | llm: gpt-4o-mini


Thank you for your question! While I don't have specific information about Sahiloan's pricing structure, many loan advisory platforms typically offer free consultations or services to help customers find the right loan options. I recommend checking the Sahiloan website or contacting their customer service directly for the most accurate and detailed information regarding any fees or costs associated with their services. If you have any other questions, feel free to ask!


❌ WITHOUT CONTEXT (Direct LLM - No RAG):
--------------------------------------------------------------------------------
Sahiloan is a financial service that typically offers loans and financial products. Whether it is free for customers can depend on various factors, such as the specific services being used, any associated fees, or interest rates on loans. Generally, while applying for loans may not have direct fees, there could be costs associated with borrowing, such as interest rates or processing fees. It's best 

## 8. Test Complete Flow (Intent Router → General Agent)

In [7]:
# Test the complete flow: Intent Router → General Agent
queries = [
    "What is Sahiloan?",
    "How is EMI calculated?",
    "Can you help me with documentation?"
]

print("="*80)
print("COMPLETE FLOW TEST (Intent Router → Agent)")
print("="*80)

for query in queries:
    print(f"\n🔍 Query: {query}")
    print("-"*80)
    
    # Step 1: Intent Router
    state = ChatState(messages=[{"role": "user", "content": query}])
    state = nodes.intent_router(state)
    route = state.get("route_to")
    print(f"   Route determined: {route}")
    
    # Step 2: Execute appropriate agent
    if route == "general_agent":
        state = nodes.general_agent(state)
        response = state["messages"][-1]["content"]
        print(f"\n   Response: {response[:200]}...")
    else:
        print(f"   ⚠️ Would route to {route} (not implemented yet)")
    
    print()

2026-01-26 16:58:44.260 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:31 - intent_router_started
2026-01-26 16:58:44.261 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:35 - intent_router_user_input: What is Sahiloan?


COMPLETE FLOW TEST (Intent Router → Agent)

🔍 Query: What is Sahiloan?
--------------------------------------------------------------------------------


2026-01-26 16:58:44.876 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:49 - intent_router_llm_response: general_agent
2026-01-26 16:58:44.877 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:60 - intent_router_completed | route: general_agent | latency: 616.63ms | llm: gpt-4o-mini
2026-01-26 16:58:44.877 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:73 - general_agent_started
2026-01-26 16:58:44.877 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:77 - general_agent_query: What is Sahiloan?
2026-01-26 16:58:44.878 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:80 - Fetching context from Pinecone...


   Route determined: general_agent


2026-01-26 16:58:46.381 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:96 - Retrieved 0 relevant chunks
2026-01-26 16:58:46.382 | WARNING  | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:114 - No relevant context found in Pinecone
2026-01-26 16:58:48.379 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:135 - general_agent_completed | chunks_used: 0 | latency: 3501.94ms | llm: gpt-4o-mini
2026-01-26 16:58:48.381 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:31 - intent_router_started
2026-01-26 16:58:48.381 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:35 - intent_router_user_input: How is EMI calculated?



   Response: Sahiloan is a loan advisory platform designed to assist individuals in finding the right loan options for their needs. We provide guidance and support throughout the loan process, helping customers un...


🔍 Query: How is EMI calculated?
--------------------------------------------------------------------------------


2026-01-26 16:58:48.955 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:49 - intent_router_llm_response: loan_agent
2026-01-26 16:58:48.956 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:60 - intent_router_completed | route: loan_agent | latency: 575.46ms | llm: gpt-4o-mini
2026-01-26 16:58:48.957 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:31 - intent_router_started
2026-01-26 16:58:48.957 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:35 - intent_router_user_input: Can you help me with documentation?


   Route determined: loan_agent
   ⚠️ Would route to loan_agent (not implemented yet)


🔍 Query: Can you help me with documentation?
--------------------------------------------------------------------------------


2026-01-26 16:58:50.173 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:49 - intent_router_llm_response: document_agent
2026-01-26 16:58:50.174 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:60 - intent_router_completed | route: document_agent | latency: 1217.10ms | llm: gpt-4o-mini


   Route determined: document_agent
   ⚠️ Would route to document_agent (not implemented yet)

